In [1]:
%matplotlib inline
from __future__ import division
import numpy as np
from numpy.random import rand
import matplotlib.pyplot as plt
from numba import jit
import os
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
#----------------------------------------------------------------------
##  BLOCK OF FUNCTIONS USED IN THE MAIN CODE
#----------------------------------------------------------------------
#@jit(nopython=True)
def initialstate(N):   
    ''' generates a random spin configuration for initial condition'''
    state = 2*np.random.randint(2, size=(N,N))-1
    return state

# def deltaU(lattice,i,j, J): #Calculate the change in energy due to flipping dipole at (i,j)
#     middle = lattice[i,j]
    
#     Ediff = 2*J*middle*(lattice[i+1,j+1]+lattice[i,j+1]+
#                        lattice[i-1,j]+lattice[i-1,j-1]+
#                        lattice[i,j-1]+lattice[i+1,j])
#     return Ediff

@jit(nopython=True)
def deltaU(lattice,i,j, J): #Calculate the change in energy due to flipping dipole at (i,j)
    middle = lattice[i,j]
    
    Ediff = 2*J*middle*(lattice[(i+1), (j+1)] + lattice[i,(j+1)] +
                       lattice[(i-1),j] + lattice[(i-1),(j-1)] +
                       lattice[i,(j-1)] + lattice[(i+1),j])
    return Ediff

def fix_boundaries(lattice):
    lattice[0,:]=lattice[-2,:]
    lattice[-1,:]=lattice[1,:]
    lattice[:,0]=lattice[:,-2]
    lattice[:,-1]=lattice[:,1]   
    return(lattice)

@jit(nopython=True)
def mcmove(config, beta, J, N):
    '''Monte Carlo move using Metropolis algorithm '''
    for i in range(N):
        for j in range(N):
                config[0,:]=config[-2,:]
                config[-1,:]=config[1,:]
                config[:,0]=config[:,-2]
                config[:,-1]=config[:,1]
                a = np.random.randint(1, N-1)
                b = np.random.randint(1, N-1)
                s =  config[a, b]
                cost = deltaU(config, a, b, J)
#                 nb = (config[(i+1)%N, (j+1)%N] + config[i,(j+1)%N] +
#                        config[(i-1)%N,j] + config[(i-1)%N,(j-1)%N] +
#                        config[i,(j-1)%N] + config[(i+1)%N,j])
#                 #nb = config[(a+1)%N,b] + config[a,(b+1)%N] + config[(a-1)%N,b] + config[a,(b-1)%N]
#                 cost = J*2*s*nb
                if cost < 0:
                    s *= -1
                elif rand() < np.exp(-cost*beta):
                    s *= -1
                config[a, b] = s
    return config

# @jit(nopython=True)
# def calcEnergy(config,  J, N):
#     '''Energy of a given configuration'''
#     energy = 0
#     for i in range(len(config)):
#         for j in range(len(config)):
#             S = config[i,j]
#             nb = config[(i+1)%N, j] + config[i,(j+1)%N] + config[(i-1)%N, j] + config[i,(j-1)%N]
#             energy += -J*nb*S
#     return energy/4.


@jit(nopython=True)
def calcMag(config):
    '''Magnetization of a given configuration'''
    mag = np.sum(config)
    return mag

def makeFigSaveData( config, N, beta, folder):
    
    X, Y = np.meshgrid(range(N), range(N))
    plt.figure() 
    #plt.setp(sp.get_yticklabels(), visible=False)
    #plt.setp(sp.get_xticklabels(), visible=False)      
    plt.pcolormesh(X, Y, config, cmap=plt.cm.binary);
    # getting current axes
    a = plt.gca()

    # set visibility of x-axis as False
    xax = a.axes.get_xaxis()
    xax = xax.set_visible(False)

    # set visibility of y-axis as False
    yax = a.axes.get_yaxis()
    yax = yax.set_visible(False)
    plt.axis('tight') 
    
    fname = 'beta_'+str(beta)+'_'
    
    if not os.path.isdir('figures/'+folder): 
        # if the demo_folder2 directory is  
        # not present then create it. 
        os.makedirs('figures/'+folder) 
    plt.savefig('./figures/'+folder+'/'+str(fname)+'.png')
    np.savetxt('./'+folder+'/'+str(fname)+'.txt', config)
    #plt.show()
    plt.close()


    

In [3]:
## change these parameters for a smaller (faster) simulation 
nt      = 2000         #  number of temperature points
N_vals  = [10,20,30,40,50,60]         #  size of the lattice, N x N
eqSteps = 10000       #  number of MC sweeps for equilibration
mcSteps = 50000       #  number of MC sweeps for calculation

type_folder = 'Hex_ferro_v2'
J= 1 

  


T       = np.linspace(1.0, 5.5, nt);
#T=[1, 2, 2.269, 3, 4]
E,M,C,X = np.zeros(nt), np.zeros(nt), np.zeros(nt), np.zeros(nt)
#n1, n2  = 1.0/(mcSteps*N*N), 1.0/(mcSteps*mcSteps*N*N) 
# divide by number of samples, and by system size to get intensive values

In [ ]:
#----------------------------------------------------------------------
#  MAIN PART OF THE CODE
#----------------------------------------------------------------------
for N in N_vals:
    folder =type_folder+'/data_N_'+str(N)+'_test'
    # checking if the directory exist or not. 
    if not os.path.isdir(folder): 
        # if the demo_folder2 directory is  
        # not present then create it. 
        os.makedirs(folder) 
    
    for tt in range(nt):
        #E1 = M1 = E2 = M2 = 0
        config = initialstate(N)
        beta=1.0/T[tt]; #iT2=iT*iT;

        for i in range(eqSteps):         # equilibrate
            config = mcmove(config, beta, J, N)           # Monte Carlo moves

        for i in range(mcSteps):
            config = mcmove(config, beta, J, N)           

        makeFigSaveData(config, N, beta, folder)
